In [10]:
# ==============================================================================
# STANDALONE TEST: Team Dashboard Defense Rankings (Akamai Bypass)
# ==============================================================================
import pandas as pd
import numpy as np
import time
import json
from nba_api.stats.endpoints import teamdashboardbygeneralsplits
from nba_api.stats.library.http import NBAStatsHTTP
from nba_api.library.http import NBAHTTP
from nba_api.stats.static import teams
import warnings
warnings.filterwarnings("ignore")

# 1. Safe Header Injection
nba_headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Referer': 'https://www.nba.com/',
    'Origin': 'https://www.nba.com',
    'Accept-Language': 'en-US,en;q=0.9',
}

if getattr(NBAHTTP, 'headers', None) is None:
    NBAHTTP.headers = nba_headers
else:
    NBAHTTP.headers.update(nba_headers)

if getattr(NBAStatsHTTP, 'headers', None) is None:
    NBAStatsHTTP.headers = nba_headers
else:
    NBAStatsHTTP.headers.update(nba_headers)

# 2. Function using TeamDashboardByGeneralSplits (One-by-one requests)
def get_defense_ranks_via_dashboard(season):
    print(f"   🛡️ Fetching Opponent Defense Rankings for {season} via Individual Dashboards...")
    try:
        nba_teams = teams.get_teams()
        opp_stats_list = []

        # Loop through all 30 teams (tqdm shows progress bar)
        for team in tqdm(nba_teams, desc="Processing Teams"):
            tid = team['id']

            # This endpoint mimics a user clicking a team page
            dash = teamdashboardbygeneralsplits.TeamDashboardByGeneralSplits(
                team_id=tid,
                measure_type_detailed_defense='Opponent',
                per_mode_detailed='PerGame',
                season=season,
                timeout=60
            )

            df = dash.get_data_frames()[0]

            if not df.empty:
                row = df.iloc[0]
                opp_stats_list.append({
                    'TEAM_ID': str(tid),
                    'OPP_PTS': row.get('OPP_PTS', 115.0),
                    'OPP_REB': row.get('OPP_REB', 44.0),
                    'OPP_AST': row.get('OPP_AST', 25.0),
                    'OPP_FG3M': row.get('OPP_FG3M', 12.0)
                })

            # small sleep to avoid rate limiting
            time.sleep(0.4)

        stats = pd.DataFrame(opp_stats_list)
        if stats.empty: return {}, {}

        # 3. Calculate Local Rankings
        col_pts, col_reb, col_ast, col_3pm = 'OPP_PTS', 'OPP_REB', 'OPP_AST', 'OPP_FG3M'

        stats['CALC_RANK_PTS'] = stats[col_pts].rank(method='min', ascending=True)
        stats['CALC_RANK_REB'] = stats[col_reb].rank(method='min', ascending=True)
        stats['CALC_RANK_AST'] = stats[col_ast].rank(method='min', ascending=True)
        stats['CALC_RANK_FG3M'] = stats[col_3pm].rank(method='min', ascending=True)

        stats['PRA'] = stats[col_pts] + stats[col_reb] + stats[col_ast]
        stats['P+R'] = stats[col_pts] + stats[col_reb]
        stats['P+A'] = stats[col_pts] + stats[col_ast]
        stats['R+A'] = stats[col_reb] + stats[col_ast]

        stats['CALC_RANK_PRA'] = stats['PRA'].rank(method='min', ascending=True)
        stats['CALC_RANK_P+R'] = stats['P+R'].rank(method='min', ascending=True)
        stats['CALC_RANK_P+A'] = stats['P+A'].rank(method='min', ascending=True)
        stats['CALC_RANK_R+A'] = stats['R+A'].rank(method='min', ascending=True)

        ranks, raw_stats = {}, {}

        for _, row in stats.iterrows():
            tid = str(row['TEAM_ID'])
            ranks[tid] = {
                'PTS': int(row['CALC_RANK_PTS']), 'REB': int(row['CALC_RANK_REB']),
                'AST': int(row['CALC_RANK_AST']), 'FG3M': int(row['CALC_RANK_FG3M']),
                'PRA': int(row['CALC_RANK_PRA']), 'P+R': int(row['CALC_RANK_P+R']),
                'P+A': int(row['CALC_RANK_P+A']), 'R+A': int(row['CALC_RANK_R+A'])
            }
            raw_stats[tid] = {
                'PTS': round(float(row.get(col_pts, 115.0)), 1),
                'REB': round(float(row.get(col_reb, 44.0)), 1),
                'AST': round(float(row.get(col_ast, 25.0)), 1),
                'FG3M': round(float(row.get(col_3pm, 12.0)), 1)
            }

        print("   ✅ Success: Dashboards processed.")
        return ranks, raw_stats

    except Exception as e:
        print(f"   ❌ FAILED: {e}")
        return {}, {}

# 4. Run Test
if __name__ == "__main__":
    # Change to current season or last season
    ranks_map, raw_data = get_defense_ranks_via_dashboard('2023-24')

    if ranks_map:
        bos_id = '1610612738' # Boston Celtics
        print("\n" + "="*50)
        print("🟢 TEST RESULTS FOR BOSTON CELTICS")
        print("="*50)
        print("\n🏀 Points allowed rank (Hardest = 1):", ranks_map[bos_id]['PTS'])
        print("📊 Raw Stats Allowed:", raw_data[bos_id])
        print("="*50 + "\n")

   🛡️ Fetching Opponent Defense Rankings for 2023-24 via Individual Dashboards...


Processing Teams:   0%|          | 0/30 [01:00<?, ?it/s]

   ❌ FAILED: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=60)
